# ============================================
# MODULE 1: DATA CLEANING FOR ELECTRICAL DATASETS
# ============================================
#
# Learning Objectives:
# - Understand common data quality issues in electrical engineering datasets
# - Handle missing values appropriately for power systems data
# - Detect and treat outliers in sensor measurements
# - Perform data type conversions and formatting
# - Validate data integrity for electrical parameters
#
# Real-World Application:
# Power systems generate massive amounts of sensor data (voltage, current, frequency, power).
# This data often contains errors from sensor malfunctions, communication failures, or
# measurement noise. Clean data is essential for accurate analysis, reliable predictions,
# and safe power system operations. In this notebook, you'll learn industry-standard
# techniques for preparing electrical engineering data for machine learning.
#
# Estimated Time: 3-4 hours
# ============================================

## Section 1: Import Libraries and Setup

In [ ]:
# Import pandas for data manipulation and analysis
# Pandas is the primary library for working with tabular data in Python
import pandas as pd

# Import numpy for numerical operations and array handling
# NumPy provides efficient mathematical operations on large datasets
import numpy as np

# Import matplotlib for data visualization
# Matplotlib is the foundation for creating plots and charts
import matplotlib.pyplot as plt

# Import seaborn for enhanced statistical visualizations
# Seaborn builds on matplotlib with better default styles
import seaborn as sns

# Import datetime for handling time-based data
# Essential for time series data common in power systems
from datetime import datetime, timedelta

# Import warnings to suppress unnecessary warning messages
import warnings

# Suppress all warnings to keep output clean
# This is useful during data exploration but should be used carefully
warnings.filterwarnings('ignore')

# Set display options for pandas DataFrames
# max_columns: Show all columns when displaying dataframes
pd.set_option('display.max_columns', None)

# Set display precision to 2 decimal places for cleaner output
pd.set_option('display.precision', 2)

# Set the style for all matplotlib plots
# 'whitegrid' provides a clean background with subtle gridlines
sns.set_style('whitegrid')

# Set default figure size for all plots (width, height in inches)
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 2: Generate Realistic Power System Data

In real-world scenarios, you would load data from sensors, SCADA systems, or databases.
For this tutorial, we'll generate realistic synthetic data that simulates common power
system measurements with typical data quality issues.

In [ ]:
# Set random seed for reproducibility
# This ensures that the random data generated is the same every time you run the notebook
np.random.seed(42)

# Define the number of records (measurements) to generate
# 1000 records simulates approximately 1 week of hourly measurements
n_records = 1000

# Create a date range starting from January 1, 2023
# freq='H' means hourly frequency, typical for power system monitoring
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Generate realistic voltage data (in kV - kilovolts)
# Normal distribution with mean=230kV and std=5kV (typical for transmission lines)
# Transmission systems typically operate at 230kV, 345kV, or 500kV
voltage = np.random.normal(loc=230, scale=5, size=n_records)

# Generate realistic current data (in Amperes)
# Normal distribution with mean=400A and std=50A
# Current varies based on load demand throughout the day
current = np.random.normal(loc=400, scale=50, size=n_records)

# Generate realistic frequency data (in Hz)
# Power systems operate at 60Hz (North America) or 50Hz (Europe/Asia)
# Small variations around 60Hz are normal, but large deviations indicate problems
frequency = np.random.normal(loc=60.0, scale=0.05, size=n_records)

# Calculate power from voltage and current (simplified)
# P = V × I × √3 × power_factor (for three-phase systems)
# Using power_factor = 0.95 (typical for industrial loads)
power_factor = 0.95
power = voltage * current * np.sqrt(3) * power_factor / 1000  # Convert to MW

# Add some realistic noise to power measurements
# Sensor noise and measurement uncertainty are common in real systems
power = power + np.random.normal(0, 5, size=n_records)

# Create the main DataFrame with all measurements
df = pd.DataFrame({
    'timestamp': date_range,
    'voltage_kv': voltage,
    'current_a': current,
    'frequency_hz': frequency,
    'power_mw': power
})

print(f"Generated {n_records} power system measurements")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

## Section 3: Introduce Realistic Data Quality Issues

Real-world data is never perfect. We'll introduce common issues found in power system data:
- Missing values (sensor failures, communication dropouts)
- Outliers (measurement errors, transient events)
- Duplicate records (logging system errors)
- Invalid values (negative power, out-of-range frequency)

In [ ]:
# Introduce missing values (simulating sensor failures)
# Randomly set 5% of voltage measurements to NaN (Not a Number)
missing_indices_voltage = np.random.choice(df.index, size=int(0.05*n_records), replace=False)
df.loc[missing_indices_voltage, 'voltage_kv'] = np.nan

# Randomly set 3% of current measurements to NaN
missing_indices_current = np.random.choice(df.index, size=int(0.03*n_records), replace=False)
df.loc[missing_indices_current, 'current_a'] = np.nan

# Introduce outliers (simulating measurement errors or transient faults)
# Set 2% of voltage values to unrealistic high values (equipment malfunction)
outlier_indices_voltage = np.random.choice(df.index, size=int(0.02*n_records), replace=False)
df.loc[outlier_indices_voltage, 'voltage_kv'] = np.random.uniform(300, 400, size=len(outlier_indices_voltage))

# Set 2% of frequency values to out-of-range values (system instability)
# Normal frequency should be very close to 60Hz; values outside 59-61Hz indicate problems
outlier_indices_freq = np.random.choice(df.index, size=int(0.02*n_records), replace=False)
df.loc[outlier_indices_freq, 'frequency_hz'] = np.random.uniform(58, 62, size=len(outlier_indices_freq))

# Introduce some negative power values (invalid - power should always be positive)
# This could happen due to sensor polarity errors or data logging issues
negative_power_indices = np.random.choice(df.index, size=int(0.01*n_records), replace=False)
df.loc[negative_power_indices, 'power_mw'] = -df.loc[negative_power_indices, 'power_mw']

# Create duplicate records (simulating logging system errors)
# Duplicate 10 random records to simulate double-logging
duplicate_indices = np.random.choice(df.index, size=10, replace=False)
duplicate_rows = df.loc[duplicate_indices].copy()
df = pd.concat([df, duplicate_rows], ignore_index=True)

# Shuffle the dataframe to make duplicates non-consecutive
# This simulates real-world scenarios where duplicates aren't always adjacent
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Data quality issues introduced:")
print(f"- Total records: {len(df)}")
print(f"- Missing voltage values: {df['voltage_kv'].isna().sum()}")
print(f"- Missing current values: {df['current_a'].isna().sum()}")
print(f"- Negative power values: {(df['power_mw'] < 0).sum()}")
print(f"- Duplicate records: {df.duplicated().sum()}")

## Section 4: Initial Data Exploration

Before cleaning, we need to understand our data's structure and quality issues.

In [ ]:
# Display the first few rows of the dataset
# This gives us a quick overview of the data structure and sample values
print("First 10 rows of the dataset:")
print(df.head(10))

In [ ]:
# Get basic information about the dataset
# info() shows: column names, data types, non-null counts, and memory usage
print("\nDataset Information:")
print(df.info())

In [ ]:
# Get statistical summary of numerical columns
# describe() provides: count, mean, std, min, 25%, 50%, 75%, max
# This helps identify outliers and understand data distribution
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Check for missing values in each column
# isnull() creates a boolean mask, sum() counts True values
print("\nMissing Values Count:")
missing_counts = df.isnull().sum()
print(missing_counts)

# Calculate percentage of missing values
# This helps prioritize which columns need attention
print("\nMissing Values Percentage:")
missing_percentages = (df.isnull().sum() / len(df)) * 100
print(missing_percentages.round(2))

In [ ]:
# Visualize missing data patterns
# This heatmap shows where missing values occur in the dataset
plt.figure(figsize=(12, 6))

# Create a heatmap where missing values are highlighted
# White = data present, Dark = missing data
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Heatmap\n(Yellow = Missing Values)', fontsize=14, fontweight='bold')
plt.xlabel('Columns')
plt.ylabel('Records')
plt.tight_layout()
plt.show()

print("\nNote: Yellow bars indicate rows with missing values")

## Section 5: Handle Duplicate Records

Duplicates can skew analysis and inflate dataset size. We'll identify and remove them.

In [ ]:
# Check for duplicate rows
# duplicated() returns True for duplicate rows (keeping first occurrence)
print(f"Number of duplicate records: {df.duplicated().sum()}")

# Show some duplicate records for inspection
# subset=['timestamp'] finds duplicates based only on timestamp
print("\nExample duplicate records:")
duplicates = df[df.duplicated(subset=['timestamp'], keep=False)].sort_values('timestamp')
print(duplicates.head(10))

In [ ]:
# Remove duplicate records
# keep='first' retains the first occurrence and removes subsequent duplicates
# inplace=False returns a new DataFrame (safer than modifying in-place)
df_clean = df.drop_duplicates(subset=['timestamp'], keep='first')

# Reset index after removing duplicates
# drop=True prevents the old index from being added as a column
df_clean = df_clean.reset_index(drop=True)

print(f"Records before removing duplicates: {len(df)}")
print(f"Records after removing duplicates: {len(df_clean)}")
print(f"Duplicates removed: {len(df) - len(df_clean)}")

## Section 6: Handle Missing Values

Missing values require careful handling in power systems data. We'll use different
strategies depending on the type of measurement and the amount of missing data.

In [ ]:
# Analyze missing value patterns
print("Missing values after removing duplicates:")
print(df_clean.isnull().sum())
print("\nPercentage of missing values:")
print((df_clean.isnull().sum() / len(df_clean) * 100).round(2))

In [ ]:
# Strategy 1: Forward Fill for time series data
# For sensor data, using the last known good value is often reasonable
# This assumes measurements change gradually over time

# Sort by timestamp to ensure chronological order
df_clean = df_clean.sort_values('timestamp').reset_index(drop=True)

# Forward fill voltage measurements
# method='ffill' (forward fill) propagates last valid observation forward
# limit=3 means fill at most 3 consecutive missing values
# This prevents filling long gaps which might not be representative
df_clean['voltage_kv'] = df_clean['voltage_kv'].fillna(method='ffill', limit=3)

print("Applied forward fill to voltage_kv (limit=3)")
print(f"Remaining missing voltage values: {df_clean['voltage_kv'].isna().sum()}")

In [ ]:
# Strategy 2: Linear Interpolation for continuous measurements
# Interpolation estimates missing values based on surrounding data points
# This is suitable for gradually changing values like current

# Interpolate current measurements
# method='linear' draws a straight line between known points
# limit=5 prevents interpolating across large gaps
df_clean['current_a'] = df_clean['current_a'].interpolate(method='linear', limit=5)

print("Applied linear interpolation to current_a (limit=5)")
print(f"Remaining missing current values: {df_clean['current_a'].isna().sum()}")

In [ ]:
# Strategy 3: Fill remaining missing values with mean
# For any remaining missing values, use the column mean
# This is a last resort that maintains the overall distribution

# Fill remaining missing voltage values with column mean
voltage_mean = df_clean['voltage_kv'].mean()
df_clean['voltage_kv'].fillna(voltage_mean, inplace=True)

# Fill remaining missing current values with column mean
current_mean = df_clean['current_a'].mean()
df_clean['current_a'].fillna(current_mean, inplace=True)

print(f"Filled remaining missing values with mean")
print(f"Voltage mean: {voltage_mean:.2f} kV")
print(f"Current mean: {current_mean:.2f} A")
print("\nFinal missing value check:")
print(df_clean.isnull().sum())

### Common Mistakes

- **Dropping all missing values**: This can remove too much data, especially in time series
- **Filling with mean without considering time dependency**: Power data has temporal patterns
- **Interpolating across large gaps**: This creates unrealistic values
- **Not checking if the filling strategy makes physical sense**: Voltage can't jump arbitrarily

### Pro Tips

- Always sort time series data before forward/backward filling
- Set reasonable limits on fill operations to avoid propagating errors
- Consider domain knowledge: voltage is more stable than current
- Document your missing value strategy for reproducibility

## Section 7: Detect and Handle Outliers

Outliers in power systems can be:
1. Real events (faults, transients) - should be kept or flagged
2. Measurement errors - should be corrected or removed

We'll use statistical methods and domain knowledge to identify outliers.

In [ ]:
# Visualize data distribution to spot outliers
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Box plot for voltage
# Box plots show median, quartiles, and outliers (points beyond whiskers)
axes[0, 0].boxplot(df_clean['voltage_kv'].dropna())
axes[0, 0].set_title('Voltage Distribution (Box Plot)', fontweight='bold')
axes[0, 0].set_ylabel('Voltage (kV)')
axes[0, 0].grid(True, alpha=0.3)

# Histogram for voltage
# Histograms show the frequency distribution of values
axes[0, 1].hist(df_clean['voltage_kv'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Voltage Distribution (Histogram)', fontweight='bold')
axes[0, 1].set_xlabel('Voltage (kV)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Box plot for current
axes[1, 0].boxplot(df_clean['current_a'].dropna())
axes[1, 0].set_title('Current Distribution (Box Plot)', fontweight='bold')
axes[1, 0].set_ylabel('Current (A)')
axes[1, 0].grid(True, alpha=0.3)

# Box plot for frequency
axes[1, 1].boxplot(df_clean['frequency_hz'].dropna())
axes[1, 1].set_title('Frequency Distribution (Box Plot)', fontweight='bold')
axes[1, 1].set_ylabel('Frequency (Hz)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Outliers are visible as points outside the box plot whiskers")

In [ ]:
# Method 1: IQR (Interquartile Range) Method for outlier detection
# IQR is the range between the 25th and 75th percentiles
# Values beyond 1.5 × IQR from quartiles are considered outliers

def detect_outliers_iqr(data, column):
    """
    Detect outliers using the IQR method.
    
    Parameters:
    data: DataFrame containing the data
    column: Column name to check for outliers
    
    Returns:
    Boolean Series indicating outlier positions (True = outlier)
    """
    # Calculate the first quartile (25th percentile)
    Q1 = data[column].quantile(0.25)
    
    # Calculate the third quartile (75th percentile)
    Q3 = data[column].quantile(0.75)
    
    # Calculate the interquartile range
    IQR = Q3 - Q1
    
    # Define lower and upper bounds
    # Standard rule: 1.5 × IQR beyond quartiles
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers as values outside the bounds
    outliers = (data[column] < lower_bound) | (data[column] > upper_bound)
    
    print(f"\n{column} Outlier Detection (IQR Method):")
    print(f"Q1 (25th percentile): {Q1:.2f}")
    print(f"Q3 (75th percentile): {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Number of outliers detected: {outliers.sum()}")
    
    return outliers

# Detect outliers in voltage
voltage_outliers = detect_outliers_iqr(df_clean, 'voltage_kv')

# Detect outliers in current
current_outliers = detect_outliers_iqr(df_clean, 'current_a')

# Detect outliers in frequency
frequency_outliers = detect_outliers_iqr(df_clean, 'frequency_hz')

In [ ]:
# Method 2: Domain-specific outlier detection
# Use electrical engineering knowledge to define acceptable ranges

# Define acceptable ranges based on power system standards
# Voltage: ±10% of nominal (230 kV) is typical acceptable range
VOLTAGE_MIN = 207  # kV (230 - 10%)
VOLTAGE_MAX = 253  # kV (230 + 10%)

# Frequency: ±0.5 Hz is the typical acceptable range for grid stability
FREQ_MIN = 59.5  # Hz
FREQ_MAX = 60.5  # Hz

# Current: Should always be positive, and we set a practical maximum
CURRENT_MIN = 0    # A (cannot be negative)
CURRENT_MAX = 700  # A (based on equipment rating)

# Identify domain-specific outliers
voltage_domain_outliers = (df_clean['voltage_kv'] < VOLTAGE_MIN) | (df_clean['voltage_kv'] > VOLTAGE_MAX)
frequency_domain_outliers = (df_clean['frequency_hz'] < FREQ_MIN) | (df_clean['frequency_hz'] > FREQ_MAX)
current_domain_outliers = (df_clean['current_a'] < CURRENT_MIN) | (df_clean['current_a'] > CURRENT_MAX)

print("Domain-Specific Outlier Detection:")
print(f"\nVoltage outliers (outside {VOLTAGE_MIN}-{VOLTAGE_MAX} kV): {voltage_domain_outliers.sum()}")
print(f"Frequency outliers (outside {FREQ_MIN}-{FREQ_MAX} Hz): {frequency_domain_outliers.sum()}")
print(f"Current outliers (outside {CURRENT_MIN}-{CURRENT_MAX} A): {current_domain_outliers.sum()}")

In [ ]:
# Handle outliers by capping (clipping) to acceptable ranges
# Clipping preserves data points while bringing extreme values within bounds
# This is preferable to deletion when outliers might contain useful information

# Cap voltage values to acceptable range
# clip() replaces values below lower bound with lower bound
# and values above upper bound with upper bound
df_clean['voltage_kv'] = df_clean['voltage_kv'].clip(lower=VOLTAGE_MIN, upper=VOLTAGE_MAX)

# Cap frequency values to acceptable range
df_clean['frequency_hz'] = df_clean['frequency_hz'].clip(lower=FREQ_MIN, upper=FREQ_MAX)

# Cap current values to acceptable range
df_clean['current_a'] = df_clean['current_a'].clip(lower=CURRENT_MIN, upper=CURRENT_MAX)

print("Outliers handled by clipping to acceptable ranges")
print("\nUpdated value ranges:")
print(f"Voltage: {df_clean['voltage_kv'].min():.2f} - {df_clean['voltage_kv'].max():.2f} kV")
print(f"Frequency: {df_clean['frequency_hz'].min():.2f} - {df_clean['frequency_hz'].max():.2f} Hz")
print(f"Current: {df_clean['current_a'].min():.2f} - {df_clean['current_a'].max():.2f} A")

## Section 8: Handle Invalid Values

Some values might be logically invalid based on electrical engineering principles.

In [ ]:
# Check for negative power values
# Power should be positive (consumption) in this context
# Negative power could indicate sensor polarity errors
negative_power = df_clean['power_mw'] < 0
print(f"Negative power values found: {negative_power.sum()}")

if negative_power.sum() > 0:
    print("\nExample negative power values:")
    print(df_clean[negative_power][['timestamp', 'power_mw']].head())
    
    # Fix negative power by taking absolute value
    # This assumes the magnitude is correct but polarity is wrong
    df_clean.loc[negative_power, 'power_mw'] = df_clean.loc[negative_power, 'power_mw'].abs()
    
    print(f"\nCorrected {negative_power.sum()} negative power values")
    print(f"Power now ranges from {df_clean['power_mw'].min():.2f} to {df_clean['power_mw'].max():.2f} MW")

In [ ]:
# Recalculate power to ensure consistency
# Sometimes cleaning voltage and current creates inconsistency with power
# Recalculating ensures physical relationships are maintained

# Calculate power from cleaned voltage and current
# P = √3 × V × I × power_factor (three-phase power formula)
power_factor = 0.95
df_clean['power_mw_calculated'] = (np.sqrt(3) * df_clean['voltage_kv'] * 
                                   df_clean['current_a'] * power_factor / 1000)

# Compare original vs recalculated power
print("Power Consistency Check:")
print(f"Original power range: {df_clean['power_mw'].min():.2f} - {df_clean['power_mw'].max():.2f} MW")
print(f"Calculated power range: {df_clean['power_mw_calculated'].min():.2f} - {df_clean['power_mw_calculated'].max():.2f} MW")
print(f"\nMean absolute difference: {(df_clean['power_mw'] - df_clean['power_mw_calculated']).abs().mean():.2f} MW")

# Replace original power with calculated power for consistency
df_clean['power_mw'] = df_clean['power_mw_calculated']
df_clean = df_clean.drop('power_mw_calculated', axis=1)

print("\nPower values updated for consistency with voltage and current")

## Section 9: Data Type Validation and Conversion

In [ ]:
# Verify data types are correct
print("Current data types:")
print(df_clean.dtypes)

# Ensure timestamp is datetime type
# This enables time-based operations and sorting
if df_clean['timestamp'].dtype != 'datetime64[ns]':
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
    print("\nConverted timestamp to datetime type")

# Ensure numerical columns are float type
# Float allows decimal precision needed for measurements
numerical_columns = ['voltage_kv', 'current_a', 'frequency_hz', 'power_mw']
for col in numerical_columns:
    df_clean[col] = df_clean[col].astype('float64')

print("\nUpdated data types:")
print(df_clean.dtypes)

## Section 10: Final Validation and Quality Check

In [ ]:
# Perform comprehensive data quality checks
print("=" * 60)
print("FINAL DATA QUALITY REPORT")
print("=" * 60)

print("\n1. Dataset Size:")
print(f"   Total records: {len(df_clean)}")
print(f"   Total columns: {len(df_clean.columns)}")

print("\n2. Missing Values:")
missing_check = df_clean.isnull().sum()
if missing_check.sum() == 0:
    print("   ✓ No missing values")
else:
    print("   ✗ Missing values found:")
    print(missing_check[missing_check > 0])

print("\n3. Duplicate Records:")
duplicates_check = df_clean.duplicated().sum()
if duplicates_check == 0:
    print("   ✓ No duplicate records")
else:
    print(f"   ✗ {duplicates_check} duplicate records found")

print("\n4. Value Ranges:")
print(f"   Voltage: {df_clean['voltage_kv'].min():.2f} - {df_clean['voltage_kv'].max():.2f} kV")
print(f"   Current: {df_clean['current_a'].min():.2f} - {df_clean['current_a'].max():.2f} A")
print(f"   Frequency: {df_clean['frequency_hz'].min():.3f} - {df_clean['frequency_hz'].max():.3f} Hz")
print(f"   Power: {df_clean['power_mw'].min():.2f} - {df_clean['power_mw'].max():.2f} MW")

print("\n5. Data Types:")
for col in df_clean.columns:
    print(f"   {col}: {df_clean[col].dtype}")

print("\n6. Time Series Continuity:")
# Check if timestamps are sorted
is_sorted = df_clean['timestamp'].is_monotonic_increasing
if is_sorted:
    print("   ✓ Timestamps are in chronological order")
else:
    print("   ✗ Timestamps are not sorted")

# Check for gaps in time series
time_diff = df_clean['timestamp'].diff()
expected_diff = pd.Timedelta(hours=1)
gaps = (time_diff > expected_diff).sum()
print(f"   Time gaps detected: {gaps}")

print("\n" + "=" * 60)
print("DATA CLEANING COMPLETE")
print("=" * 60)

In [ ]:
# Visualize cleaned data
fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# Plot voltage over time
axes[0].plot(df_clean['timestamp'], df_clean['voltage_kv'], linewidth=0.8, color='blue', alpha=0.7)
axes[0].set_title('Voltage Over Time (After Cleaning)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Voltage (kV)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=230, color='r', linestyle='--', label='Nominal Voltage', linewidth=1)
axes[0].legend()

# Plot current over time
axes[1].plot(df_clean['timestamp'], df_clean['current_a'], linewidth=0.8, color='green', alpha=0.7)
axes[1].set_title('Current Over Time (After Cleaning)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Current (A)')
axes[1].grid(True, alpha=0.3)

# Plot frequency over time
axes[2].plot(df_clean['timestamp'], df_clean['frequency_hz'], linewidth=0.8, color='orange', alpha=0.7)
axes[2].set_title('Frequency Over Time (After Cleaning)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=60.0, color='r', linestyle='--', label='Nominal Frequency', linewidth=1)
axes[2].legend()

# Plot power over time
axes[3].plot(df_clean['timestamp'], df_clean['power_mw'], linewidth=0.8, color='purple', alpha=0.7)
axes[3].set_title('Power Over Time (After Cleaning)', fontsize=12, fontweight='bold')
axes[3].set_ylabel('Power (MW)')
axes[3].set_xlabel('Timestamp')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Cleaned data visualization complete")

## Section 11: Save Cleaned Data

In [ ]:
# Save the cleaned dataset for use in future analyses
# index=False prevents writing row numbers to CSV
output_path = '../datasets/power_system_data_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print(f"File size: {len(df_clean)} records × {len(df_clean.columns)} columns")

## What This Means for Electrical Engineers

### Industry Relevance:

1. **SCADA Systems**: Real-world Supervisory Control and Data Acquisition systems generate millions of measurements daily. Data cleaning is essential for reliable monitoring.

2. **Predictive Maintenance**: Clean data enables accurate detection of equipment degradation before failures occur, saving millions in downtime costs.

3. **Grid Stability**: Outliers in frequency and voltage data can indicate serious grid stability issues that require immediate attention.

4. **Energy Trading**: Accurate load forecasting (which depends on clean historical data) is worth millions in energy markets.

5. **Regulatory Compliance**: Utilities must report accurate power quality metrics to regulators. Data quality directly affects compliance.

### Key Takeaways:

- **Data quality affects model performance**: Garbage in, garbage out applies to ML
- **Domain knowledge is crucial**: Understanding acceptable ranges for electrical parameters is essential
- **Document your cleaning process**: Reproducibility and auditability are critical in engineering
- **Balance automation with validation**: Automated cleaning should be validated with domain expertise
- **Time series require special handling**: Forward filling and interpolation respect temporal dependencies

### Next Steps:

In the next notebook, we'll perform Exploratory Data Analysis (EDA) to understand patterns, correlations, and insights in our cleaned power systems data.